# Contact Force — Label & Dataset Production

Crops individual contact-region images from synthetic particle simulations and pairs each crop with its corresponding force magnitude and force angle label.

**Pipeline overview:**
1. Iterate over paired image/label subfolders
2. For each contact on each particle, rotate the image so the contact faces upward, then crop a square region centred on the contact
3. Save each crop as a numbered PNG and accumulate `[force_magnitude, force_angle]` pairs into a single `labels.npy` file

The resulting dataset (images + `labels.npy`) is consumed directly by `Contact_force_regression_TRAINING.ipynb`.

In [ ]:
import numpy as np
import cv2
import os
import math
from pathlib import Path
import matplotlib.pyplot as plt
import re


def rotate_image(image, angle_degrees):
    h, w = image.shape[:2]
    center = (w // 2, h // 2)
    M = cv2.getRotationMatrix2D(center, angle_degrees, 1.0)
    return cv2.warpAffine(image, M, (w, h), borderMode=cv2.BORDER_CONSTANT, borderValue=0)


def numerical_sort_key(path):
    numbers = re.findall(r'\d+', path.stem)
    return int(numbers[0]) if numbers else 0


def crop_top_center_square(image, center, radius):
    h, w = image.shape[:2]
    side = int(radius)
    x1 = max(center[0] - side // 2, 0)
    x2 = min(center[0] + side // 2, w)
    y1 = 0
    y2 = min(side, h)
    return image[y1:y2, x1:x2]


def process_and_save_data_across_folders(
    img_parent_dir,
    label_parent_dir,
    output_dir,
    forces_name,
    positions_name,
    angles_name,
    test_mode=False,
    save=0
):
    """
    Crop per-contact images from synthetic particle simulations and build a label array.

    For each contact on each particle:
      1. Rotate the grayscale image so the contact points upward.
      2. Crop a square region from the top-centre of the rotated image.
      3. Optionally save the crop and accumulate [force_magnitude, force_angle] labels.

    Args:
        img_parent_dir   : Root folder; integer-named subfolders contain particle images.
        label_parent_dir : Matching root folder with .npy label files per subfolder.
        output_dir       : Destination for cropped PNGs and labels.npy.
        forces_name      : Filename of the force-magnitude .npy inside each label subfolder.
        positions_name   : Filename of the contact-angle (position) .npy.
        angles_name      : Filename of the force-direction (tangential) .npy.
        test_mode        : If True, limit processing to the first 2250 items per subfolder.
        save             : If 1, write images and labels.npy to disk.
    """
    img_parent_dir   = Path(img_parent_dir)
    label_parent_dir = Path(label_parent_dir)
    output_dir       = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)

    img_subfolders   = sorted([f for f in img_parent_dir.iterdir()   if f.is_dir() and f.name.isdigit()], key=lambda x: int(x.name))
    label_subfolders = sorted([f for f in label_parent_dir.iterdir() if f.is_dir() and f.name.isdigit()], key=lambda x: int(x.name))

    count = 0
    label_list = []

    for img_folder, label_folder in zip(img_subfolders, label_subfolders):
        forces_path    = label_folder / forces_name
        positions_path = label_folder / positions_name
        angles_path    = label_folder / angles_name

        if not (forces_path.exists() and positions_path.exists() and angles_path.exists()):
            print(f"Skipping {label_folder}: missing required files.")
            continue

        forces         = np.load(forces_path)
        contact_angles = np.load(positions_path)
        force_angles   = np.load(angles_path)

        image_files = sorted(
            [f for f in img_folder.iterdir() if f.suffix.lower() in ['.png', '.jpg', '.jpeg', '.bmp', '.tif', '.tiff']],
            key=numerical_sort_key
        )

        if test_mode:
            image_files    = image_files[:2250]
            forces         = forces[:2250]
            contact_angles = contact_angles[:2250]
            force_angles   = force_angles[:2250]

        images = []
        for img_file in image_files:
            img = cv2.imread(str(img_file), cv2.IMREAD_GRAYSCALE)
            if img is None:
                print(f"Warning: Could not read {img_file}")
                continue
            images.append(img)

        if not images:
            continue

        images   = np.stack(images, axis=0)
        h, w     = images.shape[1], images.shape[2]
        center_x = w // 2
        center_y = h // 2
        radius   = min(center_x, center_y)

        for i in range(images.shape[0]):
            img = images[i]
            if img.dtype != np.uint8:
                img = (img * 255).astype(np.uint8) if img.max() <= 1.0 else img.astype(np.uint8)

            for j in range(forces.shape[1]):
                force_mag = forces[i, j]
                force_ang = force_angles[i, j]
                contact_angle_deg = contact_angles[i, j] / np.pi * 180

                if force_mag == 0:
                    continue

                rotation_needed = 90 + contact_angle_deg
                rotated_img = rotate_image(img, rotation_needed)
                cropped_img = crop_top_center_square(rotated_img, (center_x, center_y), radius)

                count += 1
                if save:
                    cv2.imwrite(str(output_dir / f"{count:05d}.png"), cropped_img)
                label_list.append([force_mag, force_ang])

    label_array = np.array(label_list)
    if save:
        np.save(output_dir / "labels.npy", label_array)

    print(f"Done! Saved {count} cropped contact images to {output_dir}")


## Run dataset production

Set the paths below to match your local data layout, then execute the cell.

| Parameter | Description |
|---|---|
| `img_parent_dir` | Root folder whose integer-named subfolders each contain particle images |
| `label_parent_dir` | Matching root folder with `.npy` label files per subfolder |
| `output_dir` | Destination for the cropped PNGs and `labels.npy` |
| `forces_name` | Filename of the force-magnitude array inside each label subfolder |
| `positions_name` | Filename of the contact-angle (position) array |
| `angles_name` | Filename of the force-direction (tangential) array |
| `test_mode` | `1` → process only the first 2250 items per subfolder (quick sanity check) |
| `save` | `1` → write images and `labels.npy` to disk; `0` → dry run |

In [25]:
process_and_save_data_across_folders(
        img_parent_dir='C:\\Users\\jcTSAI\\Desktop\\PE_Force_ML_Data\\synthImage_poly_stress_202511\\test\\',
        label_parent_dir='C:\\Users\\jcTSAI\\Desktop\\PE_Force_ML_Data\\synth_labels_poly_stress_202511\\test\\',
        output_dir='C:\\Users\\jcTSAI\\Desktop\\PE_Force_ML_Data\\synth_img_poly_stress_contact_crop_202601',
        forces_name='mags_test.npy',
        positions_name='angles_inner_test.npy',
        angles_name='angles_tang_test.npy',
        test_mode=1, save=1
    )

Done! Saved 31500 cropped contact images to C:\Users\jcTSAI\Desktop\PE_Force_ML_Data\synth_img_poly_stress_contact_crop_202601
